In [332]:
import yfinance as yf
import pandas as pd
import numpy as np

import datetime as dt

In [333]:
TICKER = "^SPX"

START_DATE, END_DATE = '2019-01-01', '2026-05-01'

DATA_COLS = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return', 'VWAP']

In [334]:
def get_ticker_data(ticker: str = TICKER, cols: list[str] = DATA_COLS, start: str = START_DATE, end: str = END_DATE) -> pd.DataFrame:
    start_date, end_date = dt.datetime.strptime(start, "%Y-%m-%d"), dt.datetime.strptime(end, "%Y-%m-%d")

    ticker = yf.Ticker(ticker)
    data = ticker.history(start=start_date, end=end_date)

    df = data.reset_index(drop=False)
    df['Date'] = df['Date'].dt.tz_localize(None)

    df['Return'] = df['Close'].pct_change()

    price_vol = df['Close'] * df['Volume']
    
    df['VWAP'] = (
        price_vol.rolling(5).sum() /
        df['Volume'].rolling(5).sum()
    )
    
    df = df[cols]
    return df.dropna()

In [335]:
def calculate_factor(df: pd.DataFrame) -> pd.DataFrame:
    df['ret_stdev'] = df['Return'].rolling(window=5).std()

    last_close = df['Close'].shift(1)

    df['upper'] = last_close * (1 + 1.5 * df['ret_stdev'])
    df['lower'] = last_close * (1 - 1.5 * df['ret_stdev'])

    df['upper_good'] = (df['upper'] >= df['Close']) # & (abs(df['upper'] - df['High']) <= 15)
    df['lower_good'] = (df['lower'] <= df['Close']) # & (abs(df['lower'] - df['Low']) <= 15)
    df['both_good'] = df['upper_good'] & df['lower_good']

    df = df.dropna()
    return df

In [336]:
d = get_ticker_data()
d = calculate_factor(d)
d

,Date,Open,High,Low,Close,Volume,Return,VWAP,ret_stdev,upper,lower,upper_good,lower_good,both_good
8,2019-01-14,2580.310059,2589.320068,2570.409912,2582.610107,3689370000,-0.005258,2586.548296,0.005602,2618.078041,2574.441979,True,True,True
9,2019-01-15,2585.100098,2613.080078,2585.100098,2610.300049,3601180000,0.010722,2593.856084,0.005937,2605.609760,2559.610455,False,True,False
10,2019-01-16,2614.750000,2625.760010,2612.679932,2616.100098,3882180000,0.002222,2600.547427,0.005893,2633.372196,2587.227901,True,True,True
11,2019-01-17,2609.280029,2645.060059,2606.360107,2635.959961,3802410000,0.007591,2608.645854,0.006313,2640.871496,2591.328699,True,True,True
12,2019-01-18,2651.270020,2675.469971,2647.580078,2670.709961,4009010000,0.013183,2624.001576,0.007365,2665.079492,2606.840430,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1837,2026-04-24,7136.479980,7168.589844,7112.819824,7165.080078,4608830000,0.007974,7115.730030,0.007579,7189.215742,7027.584062,True,True,True
1838,2026-04-27,7152.720215,7178.740234,7146.720215,7173.910156,4783750000,0.001232,7128.201414,0.007332,7243.879125,7086.281031,True,True,True
1839,2026-04-28,7133.740234,7152.520020,7115.169922,7138.799805,4900650000,-0.004894,7143.828903,0.006945,7248.639548,7099.180764,True,True,True
1840,2026-04-29,7131.609863,7145.629883,7107.859863,7135.950195,5123100000,-0.000399,7143.376059,0.005155,7193.996822,7083.602788,True,True,True


In [337]:
d['upper_good'].describe()

count     1834
unique       2
top       True
freq      1680
Name: upper_good, dtype: object

In [338]:
d['lower_good'].describe()

count     1834
unique       2
top       True
freq      1701
Name: lower_good, dtype: object